## Profile-guided freezing (supplementary)

**This experiment is not in the manuscript.** It asked whether the robustness/sensitivity profile localises where adaptation effort matters: if the decoder and refinement blocks are the condition-dependent stages, freezing the profile-robust encoder during dereverberation fine-tuning should cost little and freezing the profile-sensitive decoder should cost a lot. MUSE was fine-tuned under three regimes, each over three seeds (`full_ft`, `freeze_encoder`, `freeze_decoder`), and the resulting checkpoints were re-probed under reverberation. The answer was a null: both freezing regimes cost the same, so the profile describes a structural property of the trained network without implying that a stage is where fine-tuning quality is won. The nine checkpoints are public on Hugging Face (`yairamr/SE-Probe-models`, under `cluster-2026-07/`), and `training/train.py --freeze_arm {full_ft,freeze_encoder,freeze_decoder}` reproduces the arms.

Runtime: under 30 seconds on any device.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
TABLES = REPO / "results_tables"
print(f"Using data from: {DATA_DIR}")
print(f"Paper tables from: {TABLES}")

## Dereverberation quality per arm

Best-epoch PESQ and STOI on the held-out RIR-Mega validation subset, mean and standard deviation over the three seeds. The `freeze_encoder` minus `freeze_decoder` difference is −0.002 PESQ with a bootstrap interval that spans zero and lies entirely below the 0.05 audibility threshold, while full fine-tuning is ahead of both arms by 0.08 to 0.09 PESQ.

In [ ]:
FA = TABLES / "freeze_arms"
stats = pd.read_csv(FA / "jobA_stats.csv").set_index("arm")
runs = pd.read_csv(FA / "jobA_per_run_stats.csv")
pub = pd.read_csv(FA / "jobA_publication_stats.csv")
display(stats[["n_seeds", "peak_pesq_mean", "peak_pesq_std", "peak_stoi_mean", "peak_stoi_std", "trainable_M"]].round(4))
display(runs[["run", "arm", "seed", "peak_epoch", "peak_pesq", "peak_stoi"]])
PAPER = {"full_ft": (2.783, 0.011), "freeze_encoder": (2.697, 0.004), "freeze_decoder": (2.700, 0.003)}
for arm, (m, sd) in PAPER.items():
    print(f"{arm:15s} PESQ {stats.loc[arm, 'peak_pesq_mean']:.3f} +/- {stats.loc[arm, 'peak_pesq_std']:.3f} (recorded {m} +/- {sd})")
    assert abs(stats.loc[arm, "peak_pesq_mean"] - m) < 5e-4 and abs(stats.loc[arm, "peak_pesq_std"] - sd) < 5e-4
d = pub[(pub.metric == "pesq") & (pub.arm_a == "freeze_encoder") & (pub.arm_b == "freeze_decoder")].iloc[0]
print(f"freeze_encoder - freeze_decoder: {d['diff']:+.3f} PESQ, 95% CI [{d.diff_ci_lo:+.3f}, {d.diff_ci_hi:+.3f}], Welch p = {d.welch_p:.2f}")
assert abs(d["diff"] - (-0.002)) < 5e-4 and abs(d.diff_ci_lo - (-0.007)) < 5e-4 and abs(d.diff_ci_hi - 0.002) < 5e-4
display(pub[["metric", "arm_a", "arm_b", "diff", "diff_ci_lo", "diff_ci_hi", "cohens_d", "welch_p", "practically_significant"]].round(4))

## Re-probing the arms under reverberation

Every arm was probed on the C50 axis (780 utterances, 88-RIR AIR pool, 13 levels). The pre-fine-tuning probe reproduces the manuscript's published pretrained MUSE artifact exactly (`jobA_probe_pre_ft_vs_manuscript.csv`). `jobA_probe_stage_summary.csv` averages the normalised AUC over the four layers of each stage, per arm (mean over seeds); the frozen stages hold their pre-fine-tuning values and the adaptation shifts to the stages that were left trainable. Note that `freeze_decoder` leaves `mag_refinement` trainable and that `latent` is frozen in neither arm.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from se_probe.layers import MUSE_STAGE_LABELS, MUSE_STAGES, layout
from se_probe.plotting import OKABE_ITO, depth_axis

stg = pd.read_csv(FA / "jobA_probe_stage_summary.csv")
piv = stg.pivot(index="stage", columns="arm", values="auc").reindex(MUSE_STAGES)
piv.insert(0, "pre_ft", stg.groupby("stage").pre_auc.first().reindex(MUSE_STAGES))
frozen = stg.pivot(index="stage", columns="arm", values="frozen").reindex(MUSE_STAGES)
display(piv.round(4))

ARMS = ["full_ft", "freeze_encoder", "freeze_decoder"]
colors = {"pre_ft": OKABE_ITO["gray"], "full_ft": OKABE_ITO["blue"], "freeze_encoder": OKABE_ITO["green"], "freeze_decoder": OKABE_ITO["vermillion"]}
fig, ax = plt.subplots(figsize=(8, 3.4))
xs = np.arange(len(MUSE_STAGES))
w = 0.2
for k, arm in enumerate(["pre_ft"] + ARMS):
    vals = piv[arm].to_numpy()
    bars = ax.bar(xs + (k - 1.5) * w, vals, w, color=colors[arm], label=arm)
    if arm in frozen.columns:
        for b_, fz in zip(bars, frozen[arm].to_numpy()):
            if fz:
                b_.set_hatch("///")
                b_.set_edgecolor("k")
ax.set_xticks(xs)
ax.set_xticklabels([MUSE_STAGE_LABELS[s] for s in MUSE_STAGES])
ax.set_ylim(0.85, 1.0)
ax.set_ylabel("stage-mean normalized AUC (C50)")
ax.set_title("Per-arm robustness by stage (hatched = frozen in that arm)")
ax.legend(loc="lower left", ncol=4, fontsize=8)
plt.tight_layout();

In [ ]:
per = pd.read_csv(FA / "jobA_probe_per_layer.csv")
pre = per[per.tag == "pre_ft"].sort_values("depth")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, arm in zip(axes, ARMS):
    t = per[per.tag == f"{arm}__seed1234"].sort_values("depth")
    x = np.arange(len(t))
    depth_axis(ax, layout("muse"))
    ax.plot(x, pre.auc.to_numpy(), marker="s", ls="--", mfc="white", color=OKABE_ITO["gray"], label="pre-FT")
    ax.fill_between(x, t.auc_lo, t.auc_hi, color=colors[arm], alpha=0.25, lw=0)
    ax.plot(x, t.auc, marker="o", color=colors[arm], label=f"{arm} (seed 1234)")
    fz = t.frozen_in_arm.to_numpy()
    if fz.any():
        lo, hi = x[fz].min() - 0.5, x[fz].max() + 0.5
        ax.axvspan(lo, hi, color="k", alpha=0.06, lw=0)
        ax.text((lo + hi) / 2, 0.86, "frozen", ha="center", fontsize=8)
    ax.set_title(arm)
    ax.legend(loc="lower left", fontsize=8)
axes[0].set_ylabel("normalized AUC (C50)")
fig.suptitle("Per-layer robustness after fine-tuning, with 95% utterance-bootstrap intervals")
plt.tight_layout();